In [13]:
import pandas as pd
import sqlite3

In [14]:
df = pd.read_csv("../data/cleaned_ecommerce_data.csv")
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   str    
 1   Date             1200 non-null   str    
 2   CustomerID       1200 non-null   str    
 3   Product          1200 non-null   str    
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   str    
 7   PaymentMethod    1200 non-null   str    
 8   OrderStatus      1200 non-null   str    
 9   TrackingNumber   1200 non-null   str    
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       1200 non-null   str    
 12  ReferralSource   1200 non-null   str    
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), str(10)
memory usage: 131.4 KB


In [15]:
conn = sqlite3.connect("ecommerce.db")
df.to_sql(
    "ecommerce_orders",
    conn,
    if_exists="replace",
    index=False
)

1200

In [16]:
query = """
SELECT *
FROM ecommerce_orders
LIMIT 5;
"""

pd.read_sql(query, conn)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


# Basic SQL Aggregation Analysis

In [26]:
query = """
SELECT
    COUNT(*) AS total_orders
FROM ecommerce_orders;
"""

pd.read_sql(query, conn)


,total_orders
0,1200


In [27]:

query = """
SELECT
    ROUND(SUM(TotalPrice), 2) AS total_revenue
FROM ecommerce_orders;
"""

pd.read_sql(query, conn)


,total_revenue
0,1264761.96


In [28]:

query = """
SELECT
    ROUND(AVG(TotalPrice), 2) AS average_order_value
FROM ecommerce_orders;
"""

pd.read_sql(query, conn)


,average_order_value
0,1053.97


In [29]:

query = """
SELECT
    OrderID,
    Product,
    Quantity,
    TotalPrice
FROM ecommerce_orders
WHERE TotalPrice >=3000
ORDER BY TotalPrice DESC;
"""

pd.read_sql(query, conn)


,OrderID,Product,Quantity,TotalPrice
0,ORD200789,Tablet,5,3456.40
1,ORD201122,Monitor,5,3390.95
2,ORD200632,Laptop,5,3390.80
3,ORD200469,Chair,5,3384.90
4,ORD200328,Tablet,5,3370.20
5,ORD200107,Printer,5,3353.75
6,ORD200326,Laptop,5,3352.40
7,ORD201065,Printer,5,3334.00
8,ORD201031,Phone,5,3322.55
9,ORD200463,Laptop,5,3313.90


In [30]:

query = """
SELECT
    Product,
    ROUND(SUM(TotalPrice), 2) AS revenue
FROM ecommerce_orders
GROUP BY Product
ORDER BY revenue DESC;
"""

pd.read_sql(query, conn)


,Product,revenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95
4,Monitor,175651.41
5,Desk,167459.93
6,Phone,151722.39


In [31]:

query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_orders
FROM ecommerce_orders
GROUP BY PaymentMethod
ORDER BY total_orders DESC;
"""

pd.read_sql(query, conn)

,PaymentMethod,total_orders
0,Online,258
1,Cash,246
2,Credit Card,234
3,Debit Card,232
4,Gift Card,230


## Date extraction

In [32]:
query = """
SELECT
    strftime('%m', Date) AS month,
    ROUND(SUM(TotalPrice), 2) AS revenue
FROM ecommerce_orders
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, conn)

,month,revenue
0,01,124313.23
1,02,112344.78
2,03,123840.93
3,04,109186.05
4,05,135142.59
5,06,170616.13
6,07,85784.64
7,08,86343.21
8,09,69321.65
9,10,89834.82


In [33]:
query = """
SELECT
    ReferralSource,
    COUNT(*) AS total_orders,
    ROUND(SUM(TotalPrice), 2) AS revenue
FROM ecommerce_orders
GROUP BY ReferralSource
ORDER BY revenue DESC;
"""

pd.read_sql(query, conn)

,ReferralSource,total_orders,revenue
0,Instagram,259,275285.45
1,Email,250,261808.55
2,Google,241,250441.48
3,Facebook,228,250410.90
4,Referral,222,226815.58


In [34]:
query = """
SELECT
    Product,
    ROUND(AVG(TotalPrice), 2) AS average_revenue
FROM ecommerce_orders
GROUP BY Product
HAVING average_revenue > 1000
ORDER BY average_revenue DESC;
"""

pd.read_sql(query, conn)

,Product,average_revenue
0,Laptop,1110.56
1,Chair,1098.99
2,Printer,1080.73
3,Monitor,1077.62
4,Tablet,1042.28
